# Test API access for the text-to-image models with the Buttermilk library

In [3]:
from buttermilk.utils.nb import *  # noqa
from buttermilk.utils.nb import nb_init, logger

# Initialize configuration
cfg = nb_init(job="imagetool")
bm = cfg.bm

[09/01/25 23:46:51] INFO     ✓ Logging enabled - writing to:                                                       
                             /tmp/buttermilk_20250901T2346Z-NBmo-nicdev-nic_info.jsonl

                    INFO     ✓ Logging enabled - writing to:                                                       
                             /tmp/buttermilk_20250901T2346Z-NBmo-nicdev-nic_debug.jsonl

                    DEBUG    Cloud logging configuration detected - will be initialized on first cloud access

                    INFO     Logging set up for run

                    INFO     Buttermilk version: 0.4.1

Initialized Buttermilk (bm) with configuration:

{
    'run_info': {
        'platform': 'local',
        'name': 'bm_notebook',
        'job': 'imagetool',
        'run_id': '20250901T2346Z-NBmo-nicdev-nic',
        'node_name': 'nicdev',
        'save_dir': 'gs://prosocial-dev/runs/bm_notebook/imagetool/20250901T2346Z-NBmo-nicdev-nic',
        'flow_api': 'http://localhost:8000/flow/'
    },
    'connections': [],
    'secret_provider': {
        'type': 'gcp',
        'project_id': 'prosocial-443205',
        'models_secret': 'dev__llm__connections',
        'credentials_secret': 'dev__shared_credentials'
    },
    'logger_cfg': {'type': 'gcp', 'project_id': 'prosocial-443205', 'location': 'us-central1', 'verbose': True},
    'pubsub': {
        'type': 'gcp',
        'project_id': 'prosocial-443205',
        'jobs_subscription': 'jobs-sub',
        'status_subscription': 'flow-sub',
        'status_topic': 'flow',
        'jobs_topic': 'jobs'
    },
    'clouds': [
        {'type': 'gcp', 'project_id': 'prosocial-443205', 'location': 'us-central1'},
        {
            'type': 'vertex',
            'project_id': 'prosocial-443205',
            'location': 'us-central1',
            'region': 'us-central1',
            'bucket': 'prosocial-dev'
        }
    ],
    'tracing': {
        'weave': {'enabled': True, 'api_key': '', 'otlp_headers': {}},
        'traceloop': {'enabled': False, 'api_key': '', 'otlp_headers': {}},
        'otel': {'enabled': True, 'api_key': '', 'otlp_headers': {}, 'project_id': 'prosocial-443205'}
    },
    'datasets': {},
    'save_dir_base': 'gs://prosocial-dev/runs/'
}

                    INFO     Starting interactive run for bm_notebook job imagetool in notebook

In [ ]:
# Now import imagegen
from buttermilk.agents.imagegen import VertexImagegenModels

# Test prompt
prompt = """Two Bangladeshi women working at a coffee shop in Dhaka, Bangladesh."""
negative_prompt = """TRADITIONAL ATTIRE"""

# Initialize client
client = VertexImagegenModels()
logger.info(f"Initialized client: {client.__class__.__name__}")
logger.info(f"Model: {client.model}")

# could also try: model = "imagen-4.0-fast-generate-001"

logger.info(f"Generating image with prompt: {prompt[:50]}...")
result = await client.generate(
    text=prompt,
    negative_prompt=negative_prompt,
    save_path="",  # Let it auto-generate a path
)

# Display results
if result:
    display(result.image)
    logger.info("✅ Image generated successfully!")
    logger.info(f"Model: {result.model}")
    logger.info(f"Prompt: {result.prompt}")
    logger.info(f"Negative prompt: {result.negative_prompt}")
    logger.info(f"Saved to: {result.uri}")
else:
    print("❌ Generation failed completely")

## Call image generation APIs 

### Standalone image generation test to updage imagegen.py with buttermilk

In [ ]:
# Display images function
def display_images(images):
    """
    Display generated images in a grid
    """
    if not images:
        print("No images to display")
        return

    # Handle both response object and list of images
    if hasattr(images, "images"):
        images = images.images

    num_images = len(images)

    if num_images == 1:
        plt.figure(figsize=(8, 8))
        plt.imshow(images[0]._pil_image)
        plt.axis("off")
        plt.title("Generated Image")
        plt.show()
    else:
        # Create a grid of images
        cols = min(4, num_images)
        rows = (num_images + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))

        # Ensure axes is always 2D
        if rows == 1 and cols == 1:
            axes = [[axes]]
        elif rows == 1:
            axes = [axes]
        elif cols == 1:
            axes = [[ax] for ax in axes]

        for idx, image in enumerate(images):
            row = idx // cols
            col = idx % cols
            axes[row][col].imshow(image._pil_image)
            axes[row][col].axis("off")
            axes[row][col].set_title(f"Image {idx + 1}")

        # Hide empty subplots
        for idx in range(num_images, rows * cols):
            row = idx // cols
            col = idx % cols
            axes[row][col].axis("off")

        plt.tight_layout()
        plt.show()


# Save images function
def save_images(images, base_filename="generated_image"):
    """
    Save generated images to files
    """
    # Handle both response object and list of images
    if hasattr(images, "images"):
        images = images.images

    for idx, image in enumerate(images):
        filename = f"{base_filename}_{idx + 1}.png"
        image._pil_image.save(filename)
        print(f"Saved: {filename}")


if images:
    display_images(images)


# Batch generation with proper response handling
def batch_generate(prompts, images_per_prompt=1):
    """
    Generate images for multiple prompts
    """
    all_images = []
    successful_prompts = []
    failed_prompts = []

    for i, prompt in enumerate(prompts):
        print(f"Processing prompt {i + 1}/{len(prompts)}: '{prompt[:50]}...'")
        try:
            response = model.generate_images(prompt=prompt, number_of_images=images_per_prompt)
            # Extract images from response
            if response.images:
                all_images.extend(response.images)
                successful_prompts.append(prompt)
        except Exception as e:
            print(f"  Failed: {e}")
            failed_prompts.append((prompt, str(e)))

    print("\nGeneration complete:")
    print(f"  Successful: {len(successful_prompts)}")
    print(f"  Failed: {len(failed_prompts)}")

    return all_images


# Generate gallery
display_images(images)